# Figures for Section: Signal Processing and Functional Connectivity Estimation

This notebook generates all 6 figures referenced in the FC section of the technical report:

1. **Fig. msc_example** — Coherence spectrum $C_{ij}(f)$ for a representative channel pair
2. **Fig. adjacency_heatmaps** — MSC adjacency heatmaps across bands/phases
3. **Fig. weight_distributions** — Histograms of off-diagonal $A_{ij}(\mathcal{B})$ per band
4. **Fig. network_properties** — Standard graph-theoretic descriptors
5. **Fig. surrogates** — Surrogate method illustration + null distribution
6. **Fig. surrogate_comparison** — Raw vs attenuated adjacency side-by-side

In [ ]:
# Setup
from lrgsglib.config.funcs import move_to_rootf
move_to_rootf(pathname='lrgeegfc')

from lrg_eegfc.notebook import *
path_figs = setup_notebook('figures/report_fc_section', dpi=200)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
from mpl_toolkits.axes_grid1 import make_axes_locatable
import networkx as nx
import numpy as np
from pathlib import Path

from lrg_eegfc.config.const import (
    BRAIN_BANDS, BRAIN_BANDS_NAMES, BRAIN_BAND_TEX_DICT,
    PHASE_LABELS, DEFAULT_SAMPLE_RATE, sEEG_DATAPATH,
)
from lrg_eegfc.utils.fc.msc.msc import band_average_msc
from lrg_eegfc.workflow.msc import load_msc_matrix

# Configuration
PATIENT = "Pat_02"
PHASE = "rsPre"
MSC_CACHE = Path("data/msc_cache")
OUTPUT_DIR = path_figs / PATIENT
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Consistent styling
CMAP_MATRIX = "magma"
BAND_COLORS = {
    'delta': '#4C72B0', 'theta': '#55A868', 'alpha': '#C44E52',
    'beta': '#8172B3', 'low_gamma': '#CCB974', 'high_gamma': '#64B5CD',
}

print(f"Patient: {PATIENT}")
print(f"Phase: {PHASE}")
print(f"Output: {OUTPUT_DIR}")

---
## Figure 1: `fig:msc_example` — Coherence spectrum for a channel pair

Plot $C_{ij}(f)$ for one representative channel pair, with band boundaries shaded
and the band-average $A_{ij}(\mathcal{B})$ highlighted.

In [ ]:
# Load pre-computed coherence tensor from cache
CACHE_DIR = Path("data/fc_fig_cache")
coh_path = CACHE_DIR / f"{PATIENT}_{PHASE}_coh_nperseg-1024.npz"

data = np.load(coh_path)
freqs, Coh = data["freqs"], data["Coh"]
fs = DEFAULT_SAMPLE_RATE

print(f"Loaded coherence tensor: {Coh.shape} (N x N x F)")
print(f"Frequency resolution: {freqs[1]-freqs[0]:.2f} Hz")
print(f"Frequency range: {freqs[0]:.2f} - {freqs[-1]:.1f} Hz")

In [ ]:
# Pick a representative channel pair with interesting coherence structure
# Strategy: find a pair with high coherence in some bands but not all
W_bands = band_average_msc(Coh, freqs, BRAIN_BANDS)

# Look for a pair with high beta coherence (visually interesting)
W_beta = W_bands['beta']
np.fill_diagonal(W_beta, 0)
triu_idx = np.triu_indices_from(W_beta, k=1)

# Find pair near the 95th percentile of beta coherence
beta_vals = W_beta[triu_idx]
target = np.percentile(beta_vals, 95)
best_idx = np.argmin(np.abs(beta_vals - target))
ch_i, ch_j = triu_idx[0][best_idx], triu_idx[1][best_idx]

print(f"Selected channel pair: ({ch_i}, {ch_j})")
for band in BRAIN_BANDS_NAMES:
    print(f"  {band}: A_ij = {W_bands[band][ch_i, ch_j]:.4f}")

In [ ]:
# Extract coherence spectrum for the selected pair
coh_spectrum = Coh[ch_i, ch_j, :]

fig, ax = plt.subplots(figsize=(10, 4))

# Plot coherence spectrum as thin line
ax.plot(freqs, coh_spectrum, color='0.3', lw=0.8, alpha=0.9, zorder=2)

# Shade each band and mark band-average with a horizontal segment
for band_name in BRAIN_BANDS_NAMES:
    fmin, fmax = BRAIN_BANDS[band_name]
    color = BAND_COLORS[band_name]
    tex = BRAIN_BAND_TEX_DICT[band_name]
    band_avg = W_bands[band_name][ch_i, ch_j]

    # Shade band region
    ax.axvspan(fmin, fmax, alpha=0.15, color=color, zorder=0)

    # Horizontal line at band-average value
    ax.hlines(band_avg, fmin, fmax, color=color, lw=2.5, zorder=3)

    # Label at top of band region
    mid_f = np.sqrt(fmin * fmax) if fmin > 0 else (fmin + fmax) / 2
    ax.text(mid_f, 1.02, tex, transform=ax.get_xaxis_transform(),
            ha='center', va='bottom', fontsize=11, color=color, fontweight='bold')

ax.set_xlim(0.5, 350)
ax.set_xscale('log')
ax.set_ylim(0, 1)
ax.set_xlabel('Frequency (Hz)', fontsize=12)
ax.set_ylabel(r'MSC $\, C_{ij}(f)$', fontsize=12)
ax.set_title(
    f'{PATIENT}, {PHASE} — Coherence spectrum (channels {ch_i}, {ch_j})',
    fontsize=13
)

# Custom x-ticks
ax.set_xticks([1, 4, 8, 13, 30, 80, 300])
ax.get_xaxis().set_major_formatter(ticker.ScalarFormatter())
ax.tick_params(which='minor', length=0)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
out = OUTPUT_DIR / 'fig_msc_example.pdf'
fig.savefig(out, bbox_inches='tight')
print(f"Saved: {out}")
plt.show()
plt.close(fig)

---
## Figure 2: `fig:adjacency_heatmaps` — MSC adjacency matrices

Grid of heatmaps: 6 bands (columns) x 4 phases (rows) with shared colorscale.

In [ ]:
# Load all MSC matrices (dense, no surrogates)
msc_data = {}
for band in BRAIN_BANDS_NAMES:
    msc_data[band] = {}
    for phase in PHASE_LABELS:
        matrix = load_msc_matrix(
            PATIENT, phase, band,
            cache_root=MSC_CACHE, sparsify='none', n_surrogates=0, nperseg=1024,
        )
        if matrix is not None:
            msc_data[band][phase] = matrix
            print(f"  {band:>12s} {phase:<12s} shape={matrix.shape}  mean={matrix[np.triu_indices_from(matrix,k=1)].mean():.4f}")
        else:
            print(f"  {band:>12s} {phase:<12s} MISSING")

In [ ]:
n_bands = len(BRAIN_BANDS_NAMES)
n_phases = len(PHASE_LABELS)

fig, axes = plt.subplots(
    n_phases, n_bands, figsize=(3.2 * n_bands, 3.0 * n_phases),
    constrained_layout=True,
)

for j, band in enumerate(BRAIN_BANDS_NAMES):
    for i, phase in enumerate(PHASE_LABELS):
        ax = axes[i, j]
        mat = msc_data.get(band, {}).get(phase)
        if mat is None:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax.transAxes)
            ax.set_xticks([]); ax.set_yticks([])
            continue

        im = ax.imshow(mat, cmap=CMAP_MATRIX, vmin=0, vmax=0.6, aspect='equal', interpolation='none')
        ax.set_xticks([]); ax.set_yticks([])

        # Row labels (phase) on leftmost column
        if j == 0:
            ax.set_ylabel(phase, fontsize=11, fontweight='bold')

        # Column titles (band) on top row
        if i == 0:
            ax.set_title(BRAIN_BAND_TEX_DICT[band], fontsize=14, fontweight='bold')

# Shared colorbar
cbar = fig.colorbar(im, ax=axes, shrink=0.6, pad=0.02, aspect=30)
cbar.set_label('MSC', fontsize=12)

fig.suptitle(f'{PATIENT} — MSC Adjacency Matrices', fontsize=15, fontweight='bold', y=1.02)

out = OUTPUT_DIR / 'fig_adjacency_heatmaps.pdf'
fig.savefig(out, bbox_inches='tight')
print(f"Saved: {out}")
plt.show()
plt.close(fig)

---
## Figure 3: `fig:weight_distributions` — Histograms of off-diagonal weights

One histogram per band (rsPre), showing the separation between the bulk of
near-zero weights and the tail of strong links.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
axes_flat = axes.ravel()

bins = np.linspace(0, 1, 80)

for k, band in enumerate(BRAIN_BANDS_NAMES):
    ax = axes_flat[k]
    mat = msc_data[band][PHASE]
    triu = mat[np.triu_indices_from(mat, k=1)]

    ax.hist(triu, bins=bins, color=BAND_COLORS[band], edgecolor='white', lw=0.3, alpha=0.85)
    ax.axvline(triu.mean(), color='k', ls='--', lw=1.2, label=f'mean = {triu.mean():.3f}')
    ax.axvline(np.percentile(triu, 95), color='red', ls=':', lw=1.2,
               label=f'P95 = {np.percentile(triu, 95):.3f}')

    ax.set_title(f'{BRAIN_BAND_TEX_DICT[band]}  ({band})', fontsize=12, fontweight='bold')
    ax.set_xlim(0, 1)
    ax.set_xlabel('$A_{ij}$' if k >= 3 else '')
    ax.set_ylabel('Count' if k % 3 == 0 else '')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_yscale('log')

fig.suptitle(
    f'{PATIENT}, {PHASE} — Distribution of off-diagonal MSC weights',
    fontsize=14, fontweight='bold',
)

out = OUTPUT_DIR / 'fig_weight_distributions.pdf'
fig.savefig(out, bbox_inches='tight')
print(f"Saved: {out}")
plt.show()
plt.close(fig)

---
## Figure 4: `fig:network_properties` — Graph-theoretic descriptors

For each band (rsPre): strength distribution, weighted clustering coefficient,
and summary table.

In [ ]:
import pandas as pd

# Compute graph metrics for each band/phase
# NOTE: MSC matrices are fully connected (dense), so path-based metrics
# (global efficiency) and degree-based metrics (assortativity) are trivial.
# We use metrics meaningful for dense weighted networks instead.
records = []
strength_data = {}  # band -> phase -> strength_array

for band in BRAIN_BANDS_NAMES:
    strength_data[band] = {}
    for phase in PHASE_LABELS:
        mat = msc_data.get(band, {}).get(phase)
        if mat is None:
            continue
        np.fill_diagonal(mat, 0)
        N = mat.shape[0]
        triu_idx = np.triu_indices_from(mat, k=1)
        triu_vals = mat[triu_idx]

        # Node strength: s_i = sum_j A_ij
        strengths = mat.sum(axis=1)
        strength_data[band][phase] = strengths

        # Weighted clustering coefficient
        G = nx.from_numpy_array(mat)
        cc = nx.average_clustering(G, weight='weight')

        # Strength assortativity: Pearson correlation of strengths
        # at the two endpoints of each edge, weighted by edge weight
        si = strengths[triu_idx[0]]
        sj = strengths[triu_idx[1]]
        strength_assort = np.corrcoef(si, sj)[0, 1]

        # Weight disparity (Herfindahl index):
        # Y_i = sum_j (A_ij / s_i)^2  — measures heterogeneity of connections
        # Y_i = 1/N for uniform weights, Y_i -> 1 for one dominant link
        with np.errstate(divide='ignore', invalid='ignore'):
            normed = mat / strengths[:, None]
            normed = np.nan_to_num(normed, 0)
        disparity = np.mean(np.sum(normed**2, axis=1))

        # Effective sparsity: fraction of total weight in top 5% of edges
        sorted_w = np.sort(triu_vals)[::-1]
        n_top5 = max(1, int(0.05 * len(sorted_w)))
        weight_concentration = sorted_w[:n_top5].sum() / sorted_w.sum()

        records.append({
            'band': band,
            'phase': phase,
            'mean_weight': triu_vals.mean(),
            'mean_strength': strengths.mean(),
            'std_strength': strengths.std(),
            'clustering_coeff': cc,
            'strength_assort': strength_assort,
            'disparity': disparity,
            'weight_concentration_5pct': weight_concentration,
        })

df_metrics = pd.DataFrame(records)
print(df_metrics.to_string(index=False, float_format='%.4f'))

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

# --- Panel A: Strength distributions per band (rsPre) ---
ax_str = fig.add_subplot(gs[0, :])
for k, band in enumerate(BRAIN_BANDS_NAMES):
    s = strength_data[band].get(PHASE)
    if s is not None:
        bp = ax_str.boxplot(
            s, positions=[k], widths=0.6,
            patch_artist=True, showfliers=False,
            boxprops=dict(facecolor=BAND_COLORS[band], alpha=0.7),
            medianprops=dict(color='black', lw=1.5),
        )

ax_str.set_xticks(range(len(BRAIN_BANDS_NAMES)))
ax_str.set_xticklabels([BRAIN_BAND_TEX_DICT[b] for b in BRAIN_BANDS_NAMES], fontsize=12)
ax_str.set_ylabel(r'Node strength $s_i = \sum_j A_{ij}$', fontsize=11)
ax_str.set_title(f'(a) Strength distribution per band — {PATIENT}, {PHASE}', fontsize=13, fontweight='bold')
ax_str.grid(axis='y', alpha=0.3)

# --- Panel B-D: Metrics across phases ---
metric_panels = [
    ('clustering_coeff', 'Weighted clustering coeff.', '(b)'),
    ('strength_assort', 'Strength assortativity', '(c)'),
    ('weight_concentration_5pct', 'Weight concentration (top 5%)', '(d)'),
]

phase_markers = {'rsPre': 'o', 'taskLearn': 's', 'taskTest': 'D', 'rsPost': '^'}

for p_idx, (metric_col, ylabel, panel_label) in enumerate(metric_panels):
    ax = fig.add_subplot(gs[1, p_idx])
    x = np.arange(len(BRAIN_BANDS_NAMES))

    for phase in PHASE_LABELS:
        subset = df_metrics[df_metrics['phase'] == phase]
        vals = [subset[subset['band'] == b][metric_col].values[0]
                if len(subset[subset['band'] == b]) > 0 else np.nan
                for b in BRAIN_BANDS_NAMES]
        ax.plot(x, vals, marker=phase_markers[phase], label=phase, lw=1.5, markersize=6)

    ax.set_xticks(x)
    ax.set_xticklabels([BRAIN_BAND_TEX_DICT[b] for b in BRAIN_BANDS_NAMES], fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(f'{panel_label} {ylabel}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, ncol=2)
    ax.grid(alpha=0.3)

out = OUTPUT_DIR / 'fig_network_properties.pdf'
fig.savefig(out, bbox_inches='tight')
print(f"Saved: {out}")
plt.show()
plt.close(fig)

---
## Figure 5: `fig:surrogates` — Surrogate method illustration + null distribution

Left: schematic of the circular-shift procedure.  
Right: null distribution histogram for one edge with observed value marked.

In [ ]:
# Load pre-computed surrogate null distribution from cache
BAND_FIG = 'beta'
CACHE_DIR = Path("data/fc_fig_cache")

# Find the null distribution file
null_files = sorted(CACHE_DIR.glob(f"{PATIENT}_{PHASE}_{BAND_FIG}_null_*.npz"))
assert len(null_files) > 0, f"No null distribution cached! Run: python scripts/precompute_fc_figures.py"
null_path = null_files[0]

null_data = np.load(null_path)
null_values = null_data["null_values"]
observed = float(null_data["observed"])
ch_i = int(null_data["ch_i"])
ch_j = int(null_data["ch_j"])
p_val = float(null_data["p_val"])
N_SUR_FIG = len(null_values)

print(f"Loaded null distribution from: {null_path.name}")
print(f"  Channels: ({ch_i}, {ch_j})")
print(f"  Observed MSC ({BAND_FIG}): {observed:.4f}")
print(f"  Null: mean={null_values.mean():.4f}, max={null_values.max():.4f}")
print(f"  p-value: {p_val:.4f}")
print(f"  N surrogates: {N_SUR_FIG}")

In [ ]:
fig = plt.figure(figsize=(14, 5))
gs_top = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[1, 1.2], wspace=0.3)

# ================================================================
# Left panel: Schematic of circular-shift surrogate method
# ================================================================
ax_schema = fig.add_subplot(gs_top[0])

np.random.seed(0)
t_demo = np.linspace(0, 2, 300)
sig1 = np.sin(2 * np.pi * 3 * t_demo) + 0.3 * np.sin(2 * np.pi * 10 * t_demo)
sig2 = 0.8 * np.sin(2 * np.pi * 3 * t_demo + 0.5) + 0.4 * np.cos(2 * np.pi * 10 * t_demo)

# Shifted versions
shift1 = int(len(t_demo) * 0.35)
shift2 = int(len(t_demo) * 0.7)
sig1_shifted = np.roll(sig1, shift1)
sig2_shifted = np.roll(sig2, shift2)

offset = 3.5  # vertical spacing

# Original signals
ax_schema.plot(t_demo, sig1 + offset, 'C0', lw=1.2, label=r'$X_i(t)$')
ax_schema.plot(t_demo, sig2 + offset, 'C1', lw=1.2, label=r'$X_j(t)$')
ax_schema.text(-0.15, offset, 'Original', fontsize=10, fontweight='bold',
               ha='right', va='center', transform=ax_schema.get_yaxis_transform())

# Shifted signals
ax_schema.plot(t_demo, sig1_shifted - offset, 'C0', lw=1.2, ls='--')
ax_schema.plot(t_demo, sig2_shifted - offset, 'C1', lw=1.2, ls='--')
ax_schema.text(-0.15, -offset, 'Shifted', fontsize=10, fontweight='bold',
               ha='right', va='center', transform=ax_schema.get_yaxis_transform())

# Arrow
ax_schema.annotate(
    '', xy=(1.0, -offset + 2.0), xytext=(1.0, offset - 2.0),
    arrowprops=dict(arrowstyle='->', lw=1.5, color='gray'),
)
ax_schema.text(1.15, 0, r'$\Delta_i^{(s)} \sim \mathrm{Unif}\{0,\ldots,T{-}1\}$',
               fontsize=9, ha='left', va='center', color='gray', style='italic')

ax_schema.set_xlabel('Time (s)', fontsize=11)
ax_schema.set_yticks([])
ax_schema.set_title('(a) Independent circular shifts', fontsize=12, fontweight='bold')
ax_schema.legend(loc='upper right', fontsize=9)

# ================================================================
# Right panel: Null distribution + observed value (broken x-axis)
# ================================================================
# Use a single axis with an inset to show the observed value position
ax_null = fig.add_subplot(gs_top[1])

# Plot null histogram zoomed to its natural range
null_max = null_values.max()
bins_null = np.linspace(0, null_max * 1.3, 35)
ax_null.hist(null_values, bins=bins_null, color='0.7', edgecolor='white', lw=0.5,
             density=True, label=f'Null ({N_SUR_FIG} surrogates)', zorder=2)

# Mark the observed value with an arrow from the right
ymax = ax_null.get_ylim()[1]
ax_null.annotate(
    f'Observed\n$A_{{ij}}$ = {observed:.3f}',
    xy=(null_max * 1.25, ymax * 0.5),
    xytext=(null_max * 1.25, ymax * 0.75),
    fontsize=10, fontweight='bold', color='red', ha='center',
    arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
)

# Draw a break indicator and arrow pointing right to indicate observed is far away
ax_null.axvline(observed, color='red', lw=2, ls='-', alpha=0.3, zorder=1)

# Add text box with statistics
textstr = (f'Null range: [{null_values.min():.4f}, {null_max:.4f}]\n'
           f'Observed: {observed:.3f}\n'
           f'$p$-value: {p_val:.4f}')
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax_null.text(0.97, 0.97, textstr, transform=ax_null.transAxes, fontsize=9,
             verticalalignment='top', horizontalalignment='right', bbox=props)

# Extend x-axis to show observed value position
ax_null.set_xlim(0, observed * 1.15)
ax_null.set_xlabel(f'MSC (band: {BAND_FIG})', fontsize=11)
ax_null.set_ylabel('Density', fontsize=11)
ax_null.set_title(
    f'(b) Null distribution — channels ({ch_i}, {ch_j})',
    fontsize=12, fontweight='bold',
)
ax_null.legend(fontsize=9, loc='upper left')

plt.tight_layout()
out = OUTPUT_DIR / 'fig_surrogates.pdf'
fig.savefig(out, bbox_inches='tight')
print(f"Saved: {out}")
plt.show()
plt.close(fig)

---
## Figure 6: `fig:surrogate_comparison` — Raw vs attenuated adjacency

Side-by-side heatmaps of $A(\mathcal{B})$ (dense) and $\hat{A}(\mathcal{B})$
(soft-sparsified), plus the difference, for one (patient, phase, band).

In [ ]:
# Load both dense and soft-sparsified matrices
BAND_CMP = 'beta'

A_dense = load_msc_matrix(
    PATIENT, PHASE, BAND_CMP,
    cache_root=MSC_CACHE, sparsify='none', n_surrogates=0, nperseg=1024,
)
A_soft = load_msc_matrix(
    PATIENT, PHASE, BAND_CMP,
    cache_root=MSC_CACHE, sparsify='soft', n_surrogates=100, nperseg=1024,
)

print(f"Dense:  shape={A_dense.shape}, mean={A_dense[np.triu_indices_from(A_dense,k=1)].mean():.4f}")
print(f"Soft:   shape={A_soft.shape},  mean={A_soft[np.triu_indices_from(A_soft,k=1)].mean():.4f}")

# Difference
A_diff = A_dense - A_soft
print(f"Diff:   mean={A_diff[np.triu_indices_from(A_diff,k=1)].mean():.4f}, "
      f"max={A_diff[np.triu_indices_from(A_diff,k=1)].max():.4f}")

# Fraction of edges attenuated
triu_dense = A_dense[np.triu_indices_from(A_dense, k=1)]
triu_soft = A_soft[np.triu_indices_from(A_soft, k=1)]
frac_attenuated = np.mean(triu_dense - triu_soft > 0.01)
frac_zeroed = np.mean((triu_dense > 0.01) & (triu_soft < 0.01))
print(f"Fraction of edges attenuated (>1%): {frac_attenuated:.1%}")
print(f"Fraction of edges effectively zeroed: {frac_zeroed:.1%}")

In [ ]:
fig = plt.figure(figsize=(16, 9))
gs = gridspec.GridSpec(2, 3, figure=fig, height_ratios=[1, 0.8], hspace=0.35, wspace=0.25)

vmax_shared = 0.6
triu_idx = np.triu_indices_from(A_dense, k=1)
dense_vals = A_dense[triu_idx]
soft_vals = A_soft[triu_idx]
diff_vals = dense_vals - soft_vals

# --- Top row: Heatmaps ---

# Dense
ax1 = fig.add_subplot(gs[0, 0])
im1 = ax1.imshow(A_dense, cmap=CMAP_MATRIX, vmin=0, vmax=vmax_shared,
                  aspect='equal', interpolation='none')
ax1.set_title(r'(a) Raw $A(\mathcal{B})$', fontsize=12, fontweight='bold')
ax1.set_xlabel('Channel'); ax1.set_ylabel('Channel')

# Soft
ax2 = fig.add_subplot(gs[0, 1])
im2 = ax2.imshow(A_soft, cmap=CMAP_MATRIX, vmin=0, vmax=vmax_shared,
                  aspect='equal', interpolation='none')
ax2.set_title(r'(b) Attenuated $\hat{A}(\mathcal{B})$', fontsize=12, fontweight='bold')
ax2.set_xlabel('Channel'); ax2.set_yticks([])

# Colorbar for heatmaps
cbar1 = fig.colorbar(im2, ax=[ax1, ax2], shrink=0.8, pad=0.02)
cbar1.set_label('MSC', fontsize=11)

# Scatter: A_dense vs A_soft
ax3 = fig.add_subplot(gs[0, 2])
ax3.scatter(dense_vals, soft_vals, s=1, alpha=0.3, color='0.4', rasterized=True)
ax3.plot([0, vmax_shared], [0, vmax_shared], 'r--', lw=1, label='identity')
ax3.set_xlabel(r'Raw $A_{ij}$', fontsize=11)
ax3.set_ylabel(r'Attenuated $\hat{A}_{ij}$', fontsize=11)
ax3.set_title('(c) Edge-wise comparison', fontsize=12, fontweight='bold')
ax3.set_xlim(0, vmax_shared); ax3.set_ylim(0, vmax_shared)
ax3.set_aspect('equal')
ax3.legend(fontsize=9)
# Correlation
r = np.corrcoef(dense_vals, soft_vals)[0, 1]
ax3.text(0.05, 0.92, f'$r$ = {r:.6f}', transform=ax3.transAxes, fontsize=10,
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# --- Bottom row: Difference analysis ---

# Histogram of differences
ax4 = fig.add_subplot(gs[1, 0:2])
ax4.hist(diff_vals, bins=80, color='0.6', edgecolor='white', lw=0.3, log=True)
ax4.axvline(0, color='k', ls='-', lw=0.8)
ax4.axvline(diff_vals.mean(), color='red', ls='--', lw=1.5,
            label=f'mean = {diff_vals.mean():.2e}')
ax4.set_xlabel(r'$A_{ij} - \hat{A}_{ij}$', fontsize=11)
ax4.set_ylabel('Count (log)', fontsize=11)
ax4.set_title(r'(d) Distribution of edge-wise differences', fontsize=12, fontweight='bold')
ax4.legend(fontsize=9)

# Summary statistics text box
ax5 = fig.add_subplot(gs[1, 2])
ax5.axis('off')
stats_text = (
    f"Edge-wise difference statistics\n"
    f"{'─' * 35}\n"
    f"Mean diff:      {diff_vals.mean():.2e}\n"
    f"Max diff:       {diff_vals.max():.2e}\n"
    f"Std diff:       {diff_vals.std():.2e}\n"
    f"{'─' * 35}\n"
    f"Pearson r:      {r:.6f}\n"
    f"Edges changed\n"
    f"  (>1% diff):   {np.mean(np.abs(diff_vals) > 0.01):.1%}\n"
    f"  (>5% diff):   {np.mean(np.abs(diff_vals) > 0.05):.1%}\n"
    f"{'─' * 35}\n"
    f"Conclusion: soft sparsification\n"
    f"has negligible effect on this\n"
    f"recording ({PATIENT}, {PHASE}, {BAND_CMP})"
)
ax5.text(0.1, 0.95, stats_text, transform=ax5.transAxes, fontsize=10,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

fig.suptitle(
    f'{PATIENT}, {PHASE}, {BAND_CMP} — Raw vs soft-sparsified MSC ($N_{{sur}}$=100)',
    fontsize=14, fontweight='bold', y=1.01,
)

out = OUTPUT_DIR / 'fig_surrogate_comparison.pdf'
fig.savefig(out, bbox_inches='tight')
print(f"Saved: {out}")
plt.show()
plt.close(fig)

---
## Summary of generated figures

| Label | File | Description |
|-------|------|-------------|
| `fig:msc_example` | `fig_msc_example.pdf` | Coherence spectrum with band regions |
| `fig:adjacency_heatmaps` | `fig_adjacency_heatmaps.pdf` | 6 bands x 4 phases MSC grid |
| `fig:weight_distributions` | `fig_weight_distributions.pdf` | Weight histograms per band |
| `fig:network_properties` | `fig_network_properties.pdf` | Graph metrics across bands/phases |
| `fig:surrogates` | `fig_surrogates.pdf` | Surrogate schematic + null distribution |
| `fig:surrogate_comparison` | `fig_surrogate_comparison.pdf` | Dense vs attenuated adjacency |

In [ ]:
# List all generated figures
print("Generated figures:")
for f in sorted(OUTPUT_DIR.resolve().glob('fig_*.pdf')):
    print(f"  {f}")